# DERMA-Agent: Comprehensive Discovery Demo

This notebook demonstrates the enhanced DERMA-Agent capabilities including:
- **Fast Discovery Engine**: Parallel hypothesis generation and testing
- **Knowledge Fabric**: Medical knowledge graph for reasoning
- **Enhanced Data Access**: Multiple TCGA cohorts with caching
- **Advanced Pathology**: Improved image segmentation and analysis
- **ML-Powered Statistics**: Random Forest, Gradient Boosting, and survival models

## Table of Contents
1. [Setup & Configuration](#1-setup--configuration)
2. [Knowledge Fabric](#2-knowledge-fabric)
3. [Data Access](#3-data-access)
4. [Fast Discovery](#4-fast-discovery)
5. [Pathology Analysis](#5-pathology-analysis)
6. [Results Dashboard](#6-results-dashboard)

## 1. Setup & Configuration

In [ ]:
import sys
import os
import json
import warnings
from pathlib import Path

# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent))

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

# DERMA-Agent imports
from agents.discovery_engine import FastDiscoveryEngine, DiscoveryConfig, run_fast_discovery
from tools.knowledge_fabric import KnowledgeFabric, create_default_knowledge_fabric, MedicalKnowledgeBuilder
from tools.enhanced_data_client import get_data_client, EXPANDED_CANCER_PROJECTS
from tools.enhanced_clinical_stats import EnhancedStatsEngine, quick_survival_summary
from tools.enhanced_pathology import (
    EnhancedWSIProcessor, NucleiSegmenter, TissueAnalyzer,
    create_synthetic_pathology_image, analyze_wsi_path
)

# Visualization settings
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
%matplotlib inline

print("✅ DERMA-Agent loaded successfully!")
print(f"📊 Available cancer types: {len(EXPANDED_CANCER_PROJECTS)}")

### 1.1 Environment Check

In [ ]:
# Check for API keys
openai_key = os.getenv("OPENAI_API_KEY")
print(f"🔑 OpenAI API Key: {'✅ Set' if openai_key else '❌ Not set (will use mock mode)'}")

# Show available cancer projects
print("\n📋 Available Cancer Projects:")
for i, (cancer, project) in enumerate(list(EXPANDED_CANCER_PROJECTS.items())[:10], 1):
    print(f"  {i:2d}. {cancer:<25} → {project}")
print(f"  ... and {len(EXPANDED_CANCER_PROJECTS) - 10} more")

## 2. Knowledge Fabric

The Knowledge Fabric is a medical knowledge graph that stores relationships between:
- **Genes/Proteins**: TP53, BRCA1, EGFR, etc.
- **Cancer Types**: Melanoma, Breast, Lung, etc.
- **Drugs/Therapies**: Pembrolizumab, Trastuzumab, etc.
- **Clinical Features**: Tumor stage, mutations, etc.
- **Pathways**: MAPK, PI3K/AKT, etc.

In [ ]:
# Create or load knowledge fabric
kg_path = Path("../data/knowledge_fabric.json")

if kg_path.exists():
    print("📚 Loading existing knowledge fabric...")
    knowledge_fabric = KnowledgeFabric.load(str(kg_path))
else:
    print("🏗️ Building knowledge fabric from scratch...")
    knowledge_fabric = create_default_knowledge_fabric(str(kg_path))

# Display statistics
stats = knowledge_fabric.get_statistics()
print(f"\n📊 Knowledge Graph Statistics:")
print(f"   Total Nodes: {stats['total_nodes']}")
print(f"   Total Edges: {stats['total_edges']}")
print(f"   Density: {stats['density']:.4f}")
print(f"\n📦 Node Types:")
for node_type, count in stats['node_types'].items():
    print(f"   • {node_type}: {count}")
print(f"\n🔗 Relation Types:")
for rel_type, count in list(stats['relation_types'].items())[:10]:
    print(f"   • {rel_type}: {count}")

### 2.1 Query the Knowledge Graph

In [ ]:
# Query: Find drugs that treat Melanoma
print("💊 Drugs that treat Melanoma:")
melanoma_drugs = []
for edge in knowledge_fabric.edges:
    if edge.relation == "TREATS" and edge.target == "Melanoma":
        drug_node = knowledge_fabric.get_node(edge.source)
        if drug_node:
            melanoma_drugs.append((drug_node.id, edge.properties))
            print(f"  • {drug_node.id}: {edge.properties}")

# Query: Find genes associated with Breast Cancer
print("\n🧬 Genes associated with Breast Cancer:")
for edge in knowledge_fabric.edges:
    if edge.relation == "MUTATED_IN" and "Breast" in edge.target:
        gene_node = knowledge_fabric.get_node(edge.source)
        if gene_node:
            print(f"  • {gene_node.id}: {edge.properties}")

# Query: Find pathways
print("\n🔄 Signaling Pathways:")
for node_id, node in knowledge_fabric.nodes.items():
    if node.label == "Pathway":
        genes = node.properties.get('genes', [])
        print(f"  • {node_id}: {', '.join(genes[:5])}{'...' if len(genes) > 5 else ''}")

### 2.2 Path Finding in Knowledge Graph

In [ ]:
# Find connection between a gene and cancer
gene = "BRAF"
cancer = "Melanoma"

print(f"🔍 Finding path from {gene} to {cancer}:")
path = knowledge_fabric.find_path(gene, cancer, max_depth=3)

if path:
    print(f"  Path found ({len(path)} nodes):")
    for i, node_id in enumerate(path):
        node = knowledge_fabric.get_node(node_id)
        if node:
            print(f"    {i+1}. {node_id} ({node.label})")
else:
    print("  No path found within depth limit")

# Get neighbors
print(f"\n🔗 Neighbors of {gene}:")
neighbors = knowledge_fabric.get_neighbors(gene)
for neighbor, edge in neighbors[:10]:
    print(f"  • {neighbor.id} ← ({edge.relation})")

## 3. Data Access

The Enhanced Data Client provides:
- **Multi-source integration**: TCGA GDC, with extensible architecture
- **Intelligent caching**: Reduces API calls and speeds up repeated queries
- **Parallel fetching**: Download multiple cohorts simultaneously
- **Automatic data preparation**: Survival-ready formatting

In [ ]:
# Initialize data client
data_client = get_data_client()

# Fetch single cohort
print("📊 Fetching Breast Cancer (TCGA-BRCA) data...")
brca_data = data_client.get_survival_analysis_ready_data("TCGA-BRCA")
brca_data = data_client.enrich_with_derived_features(brca_data)

print(f"✅ Loaded {len(brca_data)} samples")
print(f"\n📋 Columns: {list(brca_data.columns)}")

# Show data preview
brca_data.head()

In [ ]:
# Display survival summary
print(quick_survival_summary(brca_data))

# Show stage distribution
print("\n📊 Stage Distribution:")
print(brca_data['stage_group'].value_counts())

# Show gender distribution
print("\n📊 Gender Distribution:")
if 'gender' in brca_data.columns:
    print(brca_data['gender'].value_counts())

In [ ]:
# Visualize survival data
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Age distribution
if 'age_years' in brca_data.columns:
    axes[0].hist(brca_data['age_years'].dropna(), bins=30, edgecolor='black', alpha=0.7)
    axes[0].set_xlabel('Age at Diagnosis (years)')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Age Distribution')

# Survival time by vital status
if 'time' in brca_data.columns and 'event' in brca_data.columns:
    event_data = brca_data[brca_data['event'] == 1]['time'].dropna()
    censored_data = brca_data[brca_data['event'] == 0]['time'].dropna()
    
    axes[1].hist([censored_data, event_data], bins=30, 
                 label=['Censored', 'Event'], alpha=0.7, edgecolor='black')
    axes[1].set_xlabel('Time (days)')
    axes[1].set_ylabel('Count')
    axes[1].set_title('Follow-up Time Distribution')
    axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Fetch multiple cohorts for comparison
print("📊 Fetching multiple cohorts for comparison...")
cohorts = {
    "TCGA-BRCA": "Breast",
    "TCGA-SKCM": "Melanoma",
    "TCGA-LUAD": "Lung Adenocarcinoma"
}

multi_data = data_client.fetch_multiple_cohorts(list(cohorts.keys()), parallel=True)

# Prepare survival data for each
survival_data = {}
for project_id, df in multi_data.items():
    df_prepared = data_client.get_survival_analysis_ready_data(project_id)
    df_prepared = data_client.enrich_with_derived_features(df_prepared)
    survival_data[cohorts[project_id]] = df_prepared
    print(f"  ✅ {cohorts[project_id]}: {len(df_prepared)} samples")

print(f"\n📊 Total samples across cohorts: {sum(len(df) for df in survival_data.values())}")

## 4. Fast Discovery

The Fast Discovery Engine uses:
- **Parallel hypothesis testing**: Test multiple hypotheses simultaneously
- **Knowledge-guided generation**: Uses the Knowledge Fabric for relevant hypotheses
- **Self-correcting execution**: Automatically fixes code errors
- **ML-powered analysis**: Random Forest, Gradient Boosting, and advanced statistics

In [ ]:
# Configure discovery
discovery_config = DiscoveryConfig(
    max_iterations=2,
    parallel_workers=2,  # Set higher for more parallelism
    hypothesis_per_cohort=2,
    significance_threshold=0.05,
    use_knowledge_fabric=True,
    auto_correct_errors=True,
    timeout_seconds=60
)

# Initialize discovery engine
engine = FastDiscoveryEngine(discovery_config, knowledge_fabric)

print("🔬 Fast Discovery Engine initialized")
print(f"   Parallel workers: {discovery_config.parallel_workers}")
print(f"   Hypotheses per cohort: {discovery_config.hypothesis_per_cohort}")
print(f"   Using knowledge fabric: {discovery_config.use_knowledge_fabric}")

In [ ]:
# Run discovery on a single cohort
print("\n" + "="*60)
print("🚀 RUNNING DISCOVERY: Breast Cancer")
print("="*60)

results = engine.discover_single_cohort("TCGA-BRCA", "Breast Cancer")

# Display results
print("\n📊 Discovery Results:")
for i, result in enumerate(results, 1):
    print(f"\n{'='*40}")
    print(f"Hypothesis {i}:")
    print(f"  💡 {result.hypothesis[:80]}...")
    print(f"  ⏱️  Execution time: {result.execution_time:.2f}s")
    print(f"  📈 p-value: {result.p_value if result.p_value else 'N/A'}")
    print(f"  📊 Hazard Ratio: {result.hazard_ratio if result.hazard_ratio else 'N/A'}")
    print(f"  ✅ Significant: {'Yes' if result.significant else 'No'}")
    print(f"  📝 Conclusion: {result.conclusion[:100]}")

In [ ]:
# Visualize discovery results
if results:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # P-values chart
    p_values = [r.p_value for r in results if r.p_value is not None]
    labels = [f"H{i+1}" for i, r in enumerate(results) if r.p_value is not None]
    
    if p_values:
        colors = ['green' if p < 0.05 else 'red' for p in p_values]
        axes[0].bar(labels, p_values, color=colors, alpha=0.7, edgecolor='black')
        axes[0].axhline(y=0.05, color='red', linestyle='--', label='p=0.05 threshold')
        axes[0].set_ylabel('P-value')
        axes[0].set_xlabel('Hypothesis')
        axes[0].set_title('Statistical Significance of Hypotheses')
        axes[0].legend()
        axes[0].set_ylim(0, max(p_values) * 1.2 if p_values else 1)
    
    # Execution times
    exec_times = [r.execution_time for r in results]
    all_labels = [f"H{i+1}" for i in range(len(results))]
    axes[1].bar(all_labels, exec_times, color='blue', alpha=0.6, edgecolor='black')
    axes[1].set_ylabel('Time (seconds)')
    axes[1].set_xlabel('Hypothesis')
    axes[1].set_title('Execution Time per Hypothesis')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Run comprehensive discovery on multiple cohorts
print("\n" + "="*60)
print("🚀 RUNNING COMPREHENSIVE DISCOVERY")
print("="*60)

# Use the convenience function
report = run_fast_discovery(
    cancer_types=["Breast Cancer", "Skin Cancer"],
    config=DiscoveryConfig(
        parallel_workers=2,
        hypothesis_per_cohort=2,
        use_knowledge_fabric=True
    ),
    output_dir="../discoveries"
)

print("\n📊 DISCOVERY SUMMARY:")
print(f"   Total hypotheses tested: {report['total_hypotheses_tested']}")
print(f"   Significant findings: {report['significant_findings']}")
print(f"   Significance rate: {report['significance_rate']*100:.1f}%")
print(f"   Execution time: {report['execution_time_seconds']:.1f}s")

## 5. Pathology Analysis

Enhanced pathology utilities provide:
- **Advanced segmentation**: HSV, LAB, and adaptive methods
- **Nuclei detection**: Individual nuclei counting and feature extraction
- **Texture analysis**: GLCM and LBP features
- **Synthetic data**: Generate test images for development

In [ ]:
# Create synthetic pathology images
print("🔬 Creating synthetic pathology images...")

# Generate different tissue patterns
patterns = ['mixed', 'high_cellularity']
images = {}

for pattern in patterns:
    img = create_synthetic_pathology_image((512, 512), pattern)
    images[pattern] = img
    print(f"  ✅ Generated {pattern} pattern")

# Display the synthetic images
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for i, (pattern, img) in enumerate(images.items()):
    axes[i].imshow(img)
    axes[i].set_title(f'{pattern.replace("_", " ").title()} Pattern')
    axes[i].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Run nuclei segmentation
print("🔍 Running nuclei segmentation...")

segmenter = NucleiSegmenter()
analyzer = TissueAnalyzer()

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for i, (pattern, img) in enumerate(images.items()):
    # Original
    axes[i, 0].imshow(img)
    axes[i, 0].set_title(f'{pattern} - Original')
    axes[i, 0].axis('off')
    
    # HSV segmentation
    mask_hsv = segmenter.segment_nuclei(img, method='hsv')
    axes[i, 1].imshow(mask_hsv, cmap='gray')
    axes[i, 1].set_title('HSV Segmentation')
    axes[i, 1].axis('off')
    
    # LAB segmentation
    mask_lab = segmenter.segment_nuclei(img, method='lab')
    axes[i, 2].imshow(mask_lab, cmap='gray')
    axes[i, 2].set_title('LAB Segmentation')
    axes[i, 2].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Extract and analyze pathology features
print("📊 Extracting pathology features...")

for pattern, img in images.items():
    print(f"\n{pattern.upper()} PATTERN:")
    features = analyzer.analyze_tile(img)
    
    print(f"  🧬 Nuclei count: {features.nuclei_count}")
    print(f"  📊 Nuclei density: {features.nuclei_density:.2f} nuclei/mm²")
    print(f"  📏 Avg nuclei size: {features.avg_nuclei_size:.1f} pixels")
    print(f"  🔬 Cellularity: {features.cellularity:.3f}")
    print(f"  🧩 Tissue area ratio: {features.tissue_area_ratio:.3f}")
    
    if features.lymphocyte_estimate:
        print(f"  🩸 Lymphocyte estimate: {features.lymphocyte_estimate:.2%}")
    
    print(f"  🎨 Texture features:")
    for feat_name, feat_val in list(features.texture_features.items())[:5]:
        print(f"    • {feat_name}: {feat_val:.3f}")

In [ ]:
# Generate heatmaps
print("🗺️ Generating analysis heatmaps...")

fig, axes = plt.subplots(2, 2, figsize=(12, 12))

for i, (pattern, img) in enumerate(images.items()):
    # Get nuclei mask
    mask = segmenter.segment_nuclei(img)
    
    # Density heatmap
    heatmap = analyzer.generate_heatmap(img, mask, feature_type='density')
    axes[i, 0].imshow(heatmap)
    axes[i, 0].set_title(f'{pattern} - Density Heatmap')
    axes[i, 0].axis('off')")
    
    # Nuclei overlay
    nuclei_overlay = analyzer.generate_heatmap(img, mask, feature_type='nuclei')
    axes[i, 1].imshow(nuclei_overlay)
    axes[i, 1].set_title(f'{pattern} - Nuclei Overlay')
    axes[i, 1].axis('off')

plt.tight_layout()
plt.show()

## 6. Results Dashboard

Generate a comprehensive visualization of all discovery results.

In [ ]:
# Load discovery results
import glob

discovery_files = glob.glob('../discoveries/discovery_ledger_*.json')

if discovery_files:
    latest_file = max(discovery_files, key=os.path.getctime)
    print(f"📂 Loading discovery results from: {latest_file}")
    
    with open(latest_file, 'r') as f:
        ledger = json.load(f)
    
    print(f"✅ Loaded {len(ledger)} discovery entries")
    
    # Summary by project
    projects = {}
    for entry in ledger:
        pid = entry.get('project_id', 'unknown')
        if pid not in projects:
            projects[pid] = []
        projects[pid].append(entry)
    
    print("\n📊 Results by Cohort:")
    for pid, entries in projects.items():
        sig_count = sum(1 for e in entries if e.get('significant'))
        print(f"  • {pid}: {len(entries)} tested, {sig_count} significant ({100*sig_count/len(entries):.0f}%)")
else:
    print("⚠️ No discovery files found. Run discovery first.")
    ledger = []

In [ ]:
# Create comprehensive dashboard
if ledger:
    fig = plt.figure(figsize=(16, 10))
    gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)
    
    # 1. P-value distribution
    ax1 = fig.add_subplot(gs[0, 0])
    p_values = [e.get('p_value') for e in ledger if e.get('p_value') is not None]
    if p_values:
        ax1.hist(p_values, bins=20, edgecolor='black', alpha=0.7, color='steelblue')
        ax1.axvline(x=0.05, color='red', linestyle='--', linewidth=2, label='p=0.05')
        ax1.set_xlabel('P-value')
        ax1.set_ylabel('Count')
        ax1.set_title('Distribution of P-values')
        ax1.legend()
    
    # 2. Significance by cohort
    ax2 = fig.add_subplot(gs[0, 1])
    cohort_stats = {}
    for entry in ledger:
        pid = entry.get('project_id', 'unknown')
        if pid not in cohort_stats:
            cohort_stats[pid] = {'total': 0, 'significant': 0}
        cohort_stats[pid]['total'] += 1
        if entry.get('significant'):
            cohort_stats[pid]['significant'] += 1
    
    cohorts = list(cohort_stats.keys())
    rates = [cohort_stats[c]['significant']/cohort_stats[c]['total'] for c in cohorts]
    colors = ['green' if r > 0.5 else 'orange' if r > 0 else 'red' for r in rates]
    ax2.bar(range(len(cohorts)), rates, color=colors, alpha=0.7, edgecolor='black')
    ax2.set_xticks(range(len(cohorts)))
    ax2.set_xticklabels(cohorts, rotation=45, ha='right')
    ax2.set_ylabel('Significance Rate')
    ax2.set_title('Discovery Success by Cohort')
    ax2.set_ylim(0, 1)
    
    # 3. Execution time distribution
    ax3 = fig.add_subplot(gs[0, 2])
    exec_times = [e.get('execution_time', 0) for e in ledger if e.get('execution_time') is not None]
    if exec_times:
        ax3.hist(exec_times, bins=15, edgecolor='black', alpha=0.7, color='coral')
        ax3.set_xlabel('Execution Time (seconds)')
        ax3.set_ylabel('Count')
        ax3.set_title('Hypothesis Testing Speed')
    
    # 4. Top hypotheses
    ax4 = fig.add_subplot(gs[1, :])
    
    # Get top findings by significance
    significant_entries = [e for e in ledger if e.get('significant')]
    significant_entries.sort(key=lambda x: x.get('p_value', 1.0))
    
    if significant_entries:
        top_entries = significant_entries[:5]
        hypotheses = [e.get('hypothesis', 'Unknown')[:50] + '...' for e in top_entries]
        p_vals = [e.get('p_value', 0) for e in top_entries]
        cohorts_top = [e.get('project_id', 'Unknown') for e in top_entries]
        
        y_pos = np.arange(len(hypotheses))
        bars = ax4.barh(y_pos, [-np.log10(p) for p in p_vals], color='darkgreen', alpha=0.7)
        ax4.set_yticks(y_pos)
        ax4.set_yticklabels([f"[{c}] {h[:40]}" for c, h in zip(cohorts_top, hypotheses)], fontsize=8)
        ax4.set_xlabel('-log10(p-value)')
        ax4.set_title('Top Significant Findings')
        ax4.invert_yaxis()
        
        # Add significance threshold line
        threshold_line = -np.log10(0.05)
        ax4.axvline(x=threshold_line, color='red', linestyle='--', alpha=0.5, label='p=0.05')
    else:
        ax4.text(0.5, 0.5, 'No significant findings yet', 
                ha='center', va='center', transform=ax4.transAxes, fontsize=14)
        ax4.set_title('Top Significant Findings')
    
    # 5. Knowledge Graph Stats
    ax5 = fig.add_subplot(gs[2, 0])
    kg_stats = knowledge_fabric.get_statistics()
    node_types = kg_stats['node_types']
    ax5.pie(node_types.values(), labels=node_types.keys(), autopct='%1.0f%%',
           startangle=90, textprops={'fontsize': 8})
    ax5.set_title('Knowledge Graph Composition')
    
    # 6. System summary
    ax6 = fig.add_subplot(gs[2, 1:])
    ax6.axis('off')
    
    summary_text = f"""
    DERMA-Agent Discovery Summary
    ============================
    
    📊 Total Hypotheses Tested: {len(ledger)}
    ✅ Significant Findings: {len([e for e in ledger if e.get('significant')])}
    📈 Overall Significance Rate: {100*len([e for e in ledger if e.get('significant')])/len(ledger) if ledger else 0:.1f}%
    
    📚 Knowledge Graph:
       • {kg_stats['total_nodes']} nodes
       • {kg_stats['total_edges']} relationships
       • {len(kg_stats['node_types'])} node types
    
    🧬 Data Sources:
       • {len(EXPANDED_CANCER_PROJECTS)} cancer cohorts available
       • TCGA data via GDC API
       • Intelligent caching enabled
    """
    
    ax6.text(0.1, 0.9, summary_text, transform=ax6.transAxes, fontsize=11,
            verticalalignment='top', fontfamily='monospace',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))
    
    plt.suptitle('DERMA-Agent: Comprehensive Discovery Dashboard', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Save dashboard
    dashboard_path = '../discoveries/dashboard.png'
    plt.savefig(dashboard_path, dpi=150, bbox_inches='tight')
    print(f"\n💾 Dashboard saved to: {dashboard_path}")

## Summary

This notebook demonstrated the enhanced DERMA-Agent capabilities:

### ✅ Completed Features

1. **Knowledge Fabric** (`tools/knowledge_fabric.py`)
   - Medical knowledge graph with 50+ nodes
   - Genes, drugs, pathways, clinical features
   - Graph traversal and semantic search

2. **Enhanced Data Client** (`tools/enhanced_data_client.py`)
   - 20+ TCGA cancer cohorts
   - Intelligent caching (1-hour TTL)
   - Parallel cohort fetching
   - Automatic survival data preparation

3. **Fast Discovery Engine** (`agents/discovery_engine.py`)
   - Parallel hypothesis testing
   - Knowledge-guided hypothesis generation
   - Self-correcting code execution
   - Configurable significance thresholds

4. **Enhanced Clinical Stats** (`tools/enhanced_clinical_stats.py`)
   - Kaplan-Meier survival analysis
   - Cox proportional hazards regression
   - Random Forest & Gradient Boosting ML
   - Feature importance analysis

5. **Advanced Pathology** (`tools/enhanced_pathology.py`)
   - HSV, LAB, adaptive segmentation
   - Individual nuclei detection
   - GLCM & LBP texture features
   - Synthetic image generation

### 🚀 Running the System

```bash
# Run discovery on a single cancer type
python -c "from agents.discovery_engine import run_fast_discovery; run_fast_discovery(['Breast Cancer'])"

# Launch the Streamlit dashboard
streamlit run app.py

# Run the comprehensive notebook
jupyter notebook notebooks/comprehensive_demo.ipynb
```

### 📊 Results

All discovery results are saved to the `discoveries/` directory:
- `discovery_ledger_*.json` - Detailed results for each hypothesis
- `discovery_report_*.json` - Summary statistics
- `knowledge_fabric.json` - Medical knowledge graph